# Task 3 - CycleGAN (Monet <-> photo): analysis notebook

This notebook **does not train anything**. It only reads files written by `train.py` and `evaluate_local.py`
(`logs/`, `checkpoints/`, `outputs/`, `submission.csv`, `full_metrics_report.csv`). Domain **A = Monet**, domain **B = photo**.
Every cell prints a `MISSING:` message instead of failing when a file it needs has not been produced yet.

Needs `matplotlib` and `pandas` in addition to `requirements.txt`. Set `SMOKE = True` below to read the smoke-run folders.

## 1. Setup and the two domains
Counts, a sample grid, and what the class imbalance means for training.

In [ ]:
import csv, json, random, re, sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from PIL import Image

SMOKE = False        # True -> read the smoke folders (train.py --smoke / evaluate_local.py --smoke)
CHOSEN_STEP = None   # which ema_step checkpoint to analyse; None = the latest one

def find_naman():
    here = Path.cwd().resolve()
    for p in [here, *here.parents]:
        if (p / "configs" / "run01.yaml").exists() and (p / "src").is_dir():
            return p
        if (p / "naman" / "configs" / "run01.yaml").exists():
            return p / "naman"
    raise FileNotFoundError("Open this notebook from inside the naman/ folder (it must contain configs/run01.yaml).")

NAMAN = find_naman()
sys.path.insert(0, str(NAMAN / "src")); sys.path.insert(0, str(NAMAN))
from config import load_config, resolve
from data import ImageFolderDataset, build_transform, list_images, load_manifest
from models import PatchDiscriminator, ResnetGenerator, count_params

cfg = load_config(NAMAN / "configs" / "run01.yaml")
SIZE, RUN, DECAY = cfg["data"]["image_size"], cfg["run_name"], cfg["train"]["decay_start"]
CK, LOGS = resolve(cfg["paths"]["checkpoint_dir"]), resolve(cfg["paths"]["log_dir"])
OUT = resolve(cfg["paths"]["sample_dir"]).parent
CSV_DIR = NAMAN
if SMOKE:
    SIZE, RUN, DECAY = 64, RUN + "_smoke", 150
    CK, LOGS, OUT = CK / "smoke", LOGS / "smoke", OUT / "smoke"
    CSV_DIR = OUT
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
TF = build_transform(SIZE, [], train=False)   # resize -> [-1, 1], no augmentation

def rel(p):
    try:
        return str(Path(p).relative_to(NAMAN))
    except ValueError:
        return str(p)

def need(path, hint=""):
    if Path(path).exists():
        return True
    print(f"MISSING: {rel(path)}  {hint}")
    return False

def show_table(df):
    try:
        display(df)
    except NameError:
        print(df)

def to_img(t):
    return ((t.clamp(-1, 1) + 1) / 2).permute(1, 2, 0).cpu().numpy()

def ckpt_list():
    found = [(int(re.search(r"ema_step(\d+)", p.name).group(1)), p) for p in CK.glob("ema_step*.pt")]
    return sorted(found)

def chosen_ckpt():
    found = ckpt_list()
    if not found:
        return None
    step = CHOSEN_STEP
    pick = LOGS / "chosen_checkpoint.txt"   # written by post_train.ps1 (the best candidate by score)
    if step is None and pick.exists():
        hit = re.search(r"ema_step(\d+)", pick.read_text(encoding="utf-8"))
        step = int(hit.group(1)) if hit else None
    if step is None:
        return found[-1][1]
    match = [p for s, p in found if s == step]
    return match[0] if match else None

_GENS = {}
def load_gens():
    p = chosen_ckpt()
    if p is None:
        print(f"MISSING: no matching ema_step*.pt in {rel(CK)}  (run train.py first)")
        return None
    if p not in _GENS:
        ck = torch.load(p, map_location=DEVICE, weights_only=True)
        mc = ck.get("config", cfg)["model"]
        gs = []
        for key in ("G_AB", "G_BA"):
            g = ResnetGenerator(mc["ngf"], mc["n_blocks"]).to(DEVICE).eval()
            g.load_state_dict(ck[key]); gs.append(g)
        _GENS[p] = tuple(gs)
    return _GENS[p]

def param_counts():
    m = cfg["model"]
    nets = {"G_AB (Monet->photo)": ResnetGenerator(m["ngf"], m["n_blocks"]),
            "G_BA (photo->Monet)": ResnetGenerator(m["ngf"], m["n_blocks"]),
            "D_A (judges Monet)": PatchDiscriminator(m["ndf"]),
            "D_B (judges photo)": PatchDiscriminator(m["ndf"])}
    return nets, {k: count_params(v) for k, v in nets.items()}

monet_dir, photo_dir = resolve(cfg["paths"]["monet_dir"]), resolve(cfg["paths"]["photo_dir"])
FILES_A = list_images(monet_dir) if monet_dir.is_dir() else []
FILES_B = list_images(photo_dir) if photo_dir.is_dir() else []
split_path = resolve(cfg["paths"]["manifest_dir"]) / f"split_{RUN}.json"
if split_path.exists():
    split = load_manifest(split_path)
    HELD_A, HELD_B = split["A"]["heldout"], split["B"]["heldout"]
    TRAIN_A, TRAIN_B = split["A"]["train"], split["B"]["train"]
else:
    print(f"MISSING: {rel(split_path)}  (created by train.py; falling back to all images)")
    HELD_A, HELD_B, TRAIN_A, TRAIN_B = FILES_A, FILES_B, FILES_A, FILES_B
print(f"device: {DEVICE} | NAMAN = {NAMAN.name}/ | run = {RUN} | image size = {SIZE}")

In [ ]:
if not (FILES_A and FILES_B):
    print(f"MISSING: image folders {rel(monet_dir)} / {rel(photo_dir)}")
else:
    na, nb = len(FILES_A), len(FILES_B)
    print(f"Monet (A): {na} images | photos (B): {nb} images | ratio 1 : {nb / na:.1f}")
    print(f"train/held-out split: A {len(TRAIN_A)}/{len(HELD_A)}, B {len(TRAIN_B)}/{len(HELD_B)}")
    steps = cfg["train"]["total_steps"]
    print(f"With 1 Monet + 1 photo per step over {steps} steps, each train Monet is seen ~{steps / len(TRAIN_A):.0f}x, "
          f"each train photo ~{steps / len(TRAIN_B):.1f}x.")
    rng = random.Random(0)
    fig, ax = plt.subplots(2, 6, figsize=(15, 5.2))
    for r, (folder, files, name) in enumerate(((monet_dir, FILES_A, "Monet"), (photo_dir, FILES_B, "photo"))):
        for c, f in enumerate(rng.sample(files, 6)):
            ax[r, c].imshow(Image.open(folder / f).convert("RGB")); ax[r, c].axis("off")
            ax[r, c].set_title(name, fontsize=9)
    plt.tight_layout(); plt.show()

**Imbalance.** There are far fewer Monet paintings than photos, and training samples one image from each domain per step.
So the Monet side is revisited many times (the discriminator `D_A` can memorise a few hundred paintings and the generator can
overfit to them), while photos are seen only a few times each. This is why the run uses DiffAugment on discriminator inputs,
an image pool, and EMA generator weights.

## 2. Architecture
Networks built straight from `models.py` with the sizes in the config.

In [ ]:
nets, counts = param_counts()
rows = []
for name, net in nets.items():
    with torch.no_grad():
        out = net(torch.zeros(1, 3, SIZE, SIZE)).shape
    rows.append({"network": name, "parameters": counts[name], f"output for 1x3x{SIZE}x{SIZE}": tuple(out)})
rows.append({"network": "generators total", "parameters": counts["G_AB (Monet->photo)"] + counts["G_BA (photo->Monet)"]})
rows.append({"network": "all four networks", "parameters": sum(counts.values())})
tbl = pd.DataFrame(rows).fillna("")
tbl["parameters"] = tbl["parameters"].map("{:,}".format)
show_table(tbl)

ResNet generator (7x7 reflect-pad conv, 2 stride-2 downsamples, residual blocks with InstanceNorm, 2 transposed-convolution upsampling
stages (kernel 3, stride 2), Tanh) and a 70x70 PatchGAN discriminator that outputs a patch map of real/fake scores.

## 3. Training curves
From `logs/history.csv` (each row is the mean over one logging interval). The dashed line marks where the LR decay starts.
For LSGAN with the 0.5 factor, a discriminator loss near 0.25 means it outputs ~0.5 on real and fake alike (balanced game).

In [ ]:
H = LOGS / "history.csv"
if need(H, "(written by train.py)"):
    h = pd.read_csv(H).drop_duplicates("step", keep="last")  # a resumed run may re-log steps; keep the latest
    panels = [("Generator adversarial loss", ["g_adv"], False), ("Discriminator losses", ["d_a", "d_b"], False),
              ("Cycle and identity loss", ["cycle", "identity"], False),
              ("Gradient norms (log)", ["grad_g", "grad_d_a", "grad_d_b"], True),
              ("nan_count (skipped steps)", ["nan_count"], False), ("Throughput (images/sec)", ["images_per_sec"], False)]
    fig, axes = plt.subplots(2, 3, figsize=(16, 7)); axes = axes.ravel()
    for ax, (title, cols, log) in zip(axes, panels):
        for c in cols:
            ax.plot(h["step"], h[c], label=c)
        ax.axvline(DECAY, ls="--", c="gray", lw=1)
        ax.set_title(title); ax.set_xlabel("step")
        if log: ax.set_yscale("log")
        if len(cols) > 1: ax.legend()
    plt.tight_layout(); plt.show()
    print(f"{len(h)} log rows, last step {int(h['step'].iloc[-1])}, nan_count {int(h['nan_count'].iloc[-1])}")

## 4. Checkpoint selection
EMA checkpoints saved by `train.py`. `CHOSEN_STEP` (first cell) picks the one analysed in sections 6-8; `None` means the one named in `logs/chosen_checkpoint.txt`
(written by `post_train.ps1`), or the latest if that file does not exist.
`evaluate_local.py` must be run on the same checkpoint for sections 5, 7 and 8 to describe it, because its outputs do not record the step.

In [ ]:
found = ckpt_list()
if not found:
    print(f"MISSING: no ema_step*.pt in {rel(CK)}  (written by train.py every ckpt_every steps)")
else:
    ch = chosen_ckpt()
    rows = [{"step": s, "file": p.name, "size_MB": round(p.stat().st_size / 2**20, 1),
             "chosen": "<-- chosen" if p == ch else ""} for s, p in found]
    show_table(pd.DataFrame(rows))
    print("chosen:", ch.name if ch else f"no checkpoint with step {CHOSEN_STEP}")

## 5. Results
Written by `evaluate_local.py`. `MiFID` here is a local approximation; Kaggle computes its own. FID/MiFID/KID: lower is better;
LPIPS is the distance between each input and its translation; content cosine is Inception-feature similarity input vs translation.

In [ ]:
for name in ("submission.csv", "full_metrics_report.csv"):
    p = CSV_DIR / name
    if need(p, "(run evaluate_local.py)"):
        print(name); show_table(pd.read_csv(p))

## 6. Cycle-consistency check
Six photos and six Monet paintings (held-out images when the split exists), shown as original -> translated -> reconstructed
with the EMA generators of the chosen checkpoint.

In [ ]:
def show_cycle(files, folder, gen, rev, title, n=6):
    ds = ImageFolderDataset(folder, files[:n], TF)
    x = torch.stack([ds[i] for i in range(len(ds))]).to(DEVICE)
    with torch.no_grad():
        fake = gen(x); rec = rev(fake)
    fig, ax = plt.subplots(3, len(x), figsize=(2.4 * len(x), 7.5), squeeze=False)
    for j in range(len(x)):
        for i, (img, lab) in enumerate(((x, "original"), (fake, "translated"), (rec, "reconstructed"))):
            ax[i, j].imshow(to_img(img[j])); ax[i, j].axis("off")
            if j == 0: ax[i, j].set_title(lab, loc="left", fontsize=10)
    fig.suptitle(title); plt.tight_layout(); plt.show()
    print(f"{title}: mean |reconstructed - original| = {(rec - x).abs().mean().item():.4f}  (range [-1, 1])")

gens = load_gens()
if gens and FILES_A and FILES_B:
    g_ab, g_ba = gens
    show_cycle(sorted(HELD_B)[:6], photo_dir, g_ba, g_ab, "photo -> Monet -> photo (B -> A -> B)")
    show_cycle(sorted(HELD_A)[:6], monet_dir, g_ab, g_ba, "Monet -> photo -> Monet (A -> B -> A)")

## 7. Failure cases
Per-image content cosine (Inception features) and LPIPS between each input and its saved prediction are recomputed here (scores are
cached in `outputs/per_image_scores_*.csv`; delete the cache after evaluating a new checkpoint). Shown per direction: the 6
**lowest content cosine** (content lost) and the 6 **highest LPIPS** (largest perceptual change). Top row = input, bottom = translation.
The pretrained Inception/AlexNet weights are used only for measuring.

In [ ]:
def per_image_scores(tag, src_folder, src_files, pred_dir):
    cache = OUT / f"per_image_scores_{tag}.csv"
    if cache.exists():
        df = pd.read_csv(cache)
        if list(df["file"]) == list(src_files):
            return df
    import lpips, evaluate_local as ev
    ext = ev.load_inception(DEVICE)
    lp = lpips.LPIPS(net="alex", verbose=False).to(DEVICE).eval()
    pred_files = [Path(f).stem + ".jpg" for f in src_files]
    cos, lpv = [], []
    with torch.no_grad():
        for xs, xf in zip(ev.make_loader(src_folder, src_files, TF), ev.make_loader(pred_dir, pred_files, TF)):
            xs, xf = xs.to(DEVICE), xf.to(DEVICE)
            lpv.append(lp(xs, xf).flatten().cpu())
            cos.append(F.cosine_similarity(ev.inception_feats(ext, xs, DEVICE), ev.inception_feats(ext, xf, DEVICE), dim=1).cpu())
    df = pd.DataFrame({"file": list(src_files), "content_cosine": torch.cat(cos).numpy(), "lpips": torch.cat(lpv).numpy()})
    df.to_csv(cache, index=False)
    return df

def show_pairs(df, folder, pred_dir, col, ascending, title, n=6):
    top = df.sort_values(col, ascending=ascending).head(n)
    fig, ax = plt.subplots(2, n, figsize=(2.4 * n, 5.2), squeeze=False)
    for j, (_, r) in enumerate(top.iterrows()):
        ax[0, j].imshow(Image.open(folder / r["file"]).convert("RGB").resize((SIZE, SIZE)))
        ax[1, j].imshow(Image.open(pred_dir / (Path(r["file"]).stem + ".jpg")))
        ax[0, j].set_title(f"{col}={r[col]:.3f}", fontsize=9)
        ax[0, j].axis("off"); ax[1, j].axis("off")
    fig.suptitle(title); plt.tight_layout(); plt.show()

MAX_FAIL_IMAGES = None   # set e.g. 500 to speed this up on a laptop
for tag, folder, files in (("A2B", monet_dir, FILES_A), ("B2A", photo_dir, FILES_B)):
    pred_dir = OUT / f"pred_{tag}"
    if not (files and need(pred_dir, "(run evaluate_local.py)")):
        continue
    n_all = len(files)
    files = [f for f in files if (pred_dir / (Path(f).stem + ".jpg")).exists()]  # e.g. after --max-images
    if not files:
        print(f"MISSING: no predictions in {rel(pred_dir)}"); continue
    if len(files) < n_all:
        print(f"{tag}: only {len(files)} of {n_all} inputs have predictions (evaluate_local.py was run with --max-images?)")
    try:
        df = per_image_scores(tag, folder, files[:MAX_FAIL_IMAGES], pred_dir)
    except ImportError as e:
        print(f"MISSING: package for scoring ({e}); install lpips and torch-fidelity"); break
    show_pairs(df, folder, pred_dir, "content_cosine", True, f"{tag}: 6 lowest content cosine")
    show_pairs(df, folder, pred_dir, "lpips", False, f"{tag}: 6 highest LPIPS")

## 8. Human audit pack
30 fixed samples (15 Monet -> photo, 15 photo -> Monet, drawn from the **held-out** images with a fixed seed). Each is a side-by-side
image (**left = input, right = translation**) named only by a blind code `S01`..`S30`, in shuffled order. The direction lives only
in the key file, which raters must not open. Existing key and rater sheets are never overwritten.

Rating scale, 1-5 (5 is best), for each of: **style** (how convincingly the output looks like the target domain), **content**
(how well the objects and layout of the input are kept), **artifacts** (5 = clean, 1 = severe distortions/checkerboards/blobs).
Two raters fill `rater1.csv` and `rater2.csv` independently.

In [ ]:
AUDIT = OUT / "human_audit"; PAIRS = AUDIT / "pairs"
KEY = AUDIT / "KEY_do_not_open_until_rating_done.csv"
AUDIT_SEED, N_PER_DIR = 2026, 15

def build_key():
    rng = random.Random(AUDIT_SEED)
    items = []
    for direction, pool in (("A2B", HELD_A), ("B2A", HELD_B)):
        items += [(direction, f) for f in rng.sample(sorted(pool), min(N_PER_DIR, len(pool)))]
    rng.shuffle(items)
    return [{"code": f"S{i + 1:02d}", "direction": d, "file": f} for i, (d, f) in enumerate(items)]

if not (FILES_A and FILES_B):
    print("MISSING: image folders (needed to choose the audit samples)")
else:
    AUDIT.mkdir(parents=True, exist_ok=True)
    if KEY.exists():
        with open(KEY, encoding="utf-8") as f:
            key = list(csv.DictReader(f))
        print(f"reusing existing key {rel(KEY)} ({len(key)} samples)")
    else:
        key = build_key()
        with open(KEY, "w", newline="", encoding="utf-8") as f:
            w = csv.DictWriter(f, fieldnames=["code", "direction", "file"]); w.writeheader(); w.writerows(key)
        print(f"wrote key {rel(KEY)} ({len(key)} samples)")
    for r in ("rater1", "rater2"):
        p = AUDIT / f"{r}.csv"
        if p.exists():
            print(f"kept existing {rel(p)}"); continue
        with open(p, "w", newline="", encoding="utf-8") as f:
            w = csv.writer(f); w.writerow(["code", "style", "content", "artifacts"])
            for k in key: w.writerow([k["code"], "", "", ""])
        print(f"wrote blank sheet {rel(p)}")
    srcs = {"A2B": monet_dir, "B2A": photo_dir}
    missing = [k["code"] for k in key if not (OUT / f"pred_{k['direction']}" / (Path(k["file"]).stem + ".jpg")).exists()]
    if missing:
        print(f"MISSING: predictions for {len(missing)} of {len(key)} audit samples in {rel(OUT)}/pred_*  "
              "(run evaluate_local.py on ALL images, i.e. without --max-images); pair images not written")
    else:
        PAIRS.mkdir(exist_ok=True)
        gap = 8
        for k in key:
            src = Image.open(srcs[k["direction"]] / k["file"]).convert("RGB").resize((SIZE, SIZE), Image.BILINEAR)
            pred = Image.open(OUT / f"pred_{k['direction']}" / (Path(k["file"]).stem + ".jpg")).convert("RGB")
            canvas = Image.new("RGB", (2 * SIZE + gap, SIZE), "white")
            canvas.paste(src, (0, 0)); canvas.paste(pred, (SIZE + gap, 0))
            canvas.save(PAIRS / f"{k['code']}.png")
        print(f"wrote {len(key)} pair images to {rel(PAIRS)}")

In [ ]:
# Run after BOTH sheets are filled (integers 1-5 in style, content, artifacts).
CRITERIA = ["style", "content", "artifacts"]

def weighted_kappa(a, b, k=5):
    # quadratic-weighted Cohen's kappa for ordinal ratings 1..k
    a, b = np.asarray(a, int) - 1, np.asarray(b, int) - 1
    O = np.zeros((k, k))
    for i, j in zip(a, b): O[i, j] += 1
    E = np.outer(O.sum(1), O.sum(0)) / O.sum()
    W = (np.arange(k)[:, None] - np.arange(k)[None, :]) ** 2 / (k - 1) ** 2
    return np.nan if (W * E).sum() == 0 else 1 - (W * O).sum() / (W * E).sum()

sheets = {}
for r in ("rater1", "rater2"):
    p = AUDIT / f"{r}.csv"
    if not need(p, "(create it with the previous cell)"):
        break
    d = pd.read_csv(p).set_index("code")
    vals = d[CRITERIA].apply(pd.to_numeric, errors="coerce")
    bad_n = int((vals.isna() | ~vals.isin([1, 2, 3, 4, 5])).sum().sum())
    if bad_n:
        print(f"{r}.csv still has {bad_n} blank or invalid scores (need integers 1-5)"); break
    sheets[r] = vals.astype(int)
if len(sheets) == 2 and need(KEY):
    r1 = sheets["rater1"]; r2 = sheets["rater2"].loc[r1.index]
    keydf = pd.read_csv(KEY).set_index("code")
    mean = pd.DataFrame({"rater1": r1.mean(), "rater2": r2.mean(), "both": pd.concat([r1, r2]).mean()})
    print("Average scores"); show_table(mean.round(2))
    both = ((r1 + r2) / 2).join(keydf["direction"])
    print("Average of the two raters by direction"); show_table(both.groupby("direction")[CRITERIA].mean().round(2))
    kap = pd.Series({c: weighted_kappa(r1[c], r2[c]) for c in CRITERIA}, name="weighted kappa (quadratic)")
    print("Inter-rater agreement"); show_table(kap.round(3).to_frame())
    print("(kappa is NaN when a rater gave the same score to every sample)")
elif len(sheets) < 2:
    print("Fill both rater sheets, then re-run this cell.")

## 9. Summary
Parameter counts from `models.py`; time and memory from `logs/run_summary.json`; throughput from `logs/history.csv`.

In [ ]:
summary = {}
_, counts = param_counts()
summary["parameters, 2 generators"] = f"{counts['G_AB (Monet->photo)'] + counts['G_BA (photo->Monet)']:,}"
summary["parameters, all 4 networks"] = f"{sum(counts.values()):,}"
RS = LOGS / "run_summary.json"
if need(RS, "(written by train.py at the end of a run)"):
    with open(RS, encoding="utf-8") as f:
        s = json.load(f)
    summary["steps"] = s["steps"]
    summary["training time"] = f"{s['total_train_seconds'] / 3600:.2f} h ({s['total_train_seconds']:.0f} s)"
    summary["peak GPU memory"] = "n/a (CPU run)" if s["peak_gpu_memory_mb"] is None else f"{s['peak_gpu_memory_mb']:.0f} MB"
    summary["device"] = s["device"]
    summary["nan_count"] = s["nan_count"]
if need(LOGS / "history.csv", "(written by train.py)"):
    ips = pd.read_csv(LOGS / "history.csv").drop_duplicates("step", keep="last")["images_per_sec"]
    summary["images/sec (mean / median)"] = f"{ips.mean():.1f} / {ips.median():.1f}"
show_table(pd.DataFrame({"value": summary}))